# NEPA - CIFAR-10 Training Notebook

This notebook provides a complete workflow for training NEPA (Next-Embedding Prediction Architecture) on CIFAR-10 using a tiny model configuration optimized for single GPU training.

**Workflow:**
1. Setup & Dependencies
2. Configuration
3. Dataset Loading
4. Pretraining
5. Fine-tuning
6. Evaluation
7. Save/Load Checkpoints

## 1. Setup & Dependencies

First, let's install the required dependencies and clone the NEPA repository.

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Clone the NEPA repository
!git clone https://github.com/mu-hashmi/nepa.git
%cd nepa

In [ ]:
# Install dependencies
!pip install -q transformers==4.56.2 datasets==3.6.0 accelerate>=0.12.0 
!pip install -q evaluate scikit-learn timm diffusers
!pip install -q wandb  # Optional: for experiment tracking

In [ ]:
# Import required libraries
import os
import torch
from datasets import load_dataset
from transformers import TrainingArguments, Trainer
from PIL import Image
import json

# Add models to path
import sys
sys.path.insert(0, '.')

from models.vit_nepa import ViTNepaConfig, ViTNepaForPreTraining, ViTNepaForImageClassification

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

Load the tiny model configuration optimized for single GPU training.

In [ ]:
# Tiny model configuration for CIFAR-10
# ~15M parameters, suitable for single GPU

PRETRAIN_CONFIG = {
    "add_pooling_layer": True,
    "architectures": ["ViTNepaForPreTraining"],
    "attention_probs_dropout_prob": 0.0,
    "drop_path_prob": 0.0,
    "hidden_act": "gelu",
    "hidden_dropout_prob": 0.0,
    "hidden_size": 384,
    "image_size": 224,
    "initializer_range": 0.02,
    "intermediate_size": 1536,
    "is_causal": True,
    "layer_norm_eps": 1e-12,
    "layerscale_value": 1e-05,
    "model_type": "vit_nepa",
    "num_attention_heads": 6,
    "num_channels": 3,
    "num_hidden_layers": 6,
    "patch_size": 14,
    "qk_norm": True,
    "qkv_bias": True,
    "rope_theta": 100.0,
    "use_gated_mlp": False
}

# CIFAR-10 labels
CIFAR10_LABELS = {
    0: "airplane", 1: "automobile", 2: "bird", 3: "cat", 4: "deer",
    5: "dog", 6: "frog", 7: "horse", 8: "ship", 9: "truck"
}

print("Configuration loaded!")
print(f"Hidden size: {PRETRAIN_CONFIG['hidden_size']}")
print(f"Num layers: {PRETRAIN_CONFIG['num_hidden_layers']}")
print(f"Num heads: {PRETRAIN_CONFIG['num_attention_heads']}")

## 3. Dataset Loading

Load CIFAR-10 dataset from HuggingFace Hub.

In [ ]:
# Load CIFAR-10 dataset
dataset = load_dataset("cifar10")

print(f"Train samples: {len(dataset['train'])}")
print(f"Test samples: {len(dataset['test'])}")
print(f"Features: {dataset['train'].features}")

In [ ]:
# Visualize some samples
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    sample = dataset['train'][i]
    ax.imshow(sample['img'])
    ax.set_title(CIFAR10_LABELS[sample['label']])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Pretraining

Pretrain the NEPA model using next-embedding prediction on CIFAR-10.

**Note:** This uses the pretraining script which expects images to be resized to 224x224.

In [ ]:
# Training hyperparameters for pretraining
PRETRAIN_ARGS = {
    "batch_size": 128,
    "learning_rate": 2e-4,
    "num_epochs": 100,  # Reduced for faster iteration
    "warmup_ratio": 0.05,
    "weight_decay": 0.05,
}

print("Pretraining hyperparameters:")
for k, v in PRETRAIN_ARGS.items():
    print(f"  {k}: {v}")

In [ ]:
# Run pretraining using the shell script
# This will take some time depending on your GPU

!python run_nepa.py \
    --config_name configs/pretrain/nepa-tiny-patch14-224-cifar10 \
    --image_processor_name configs/pretrain/nepa-tiny-patch14-224-cifar10 \
    --dataset_name cifar10 \
    --load_from_disk False \
    --do_train \
    --output_dir outputs/nepa-tiny-cifar10-pretrain \
    --num_train_epochs 100 \
    --per_device_train_batch_size 128 \
    --learning_rate 2e-4 \
    --lr_scheduler_type cosine \
    --warmup_ratio 0.05 \
    --weight_decay 0.05 \
    --logging_steps 50 \
    --save_steps 5000 \
    --seed 1337 \
    --bf16 True \
    --dataloader_num_workers 2 \
    --remove_unused_columns False \
    --report_to none

## 5. Fine-tuning

Fine-tune the pretrained model for image classification on CIFAR-10.

In [ ]:
# Fine-tuning hyperparameters
FINETUNE_ARGS = {
    "batch_size": 64,
    "learning_rate": 5e-4,
    "num_epochs": 50,  # Reduced for faster iteration
    "warmup_ratio": 0.10,
    "weight_decay": 0.05,
    "llrd": 0.65,  # Layer-wise learning rate decay
    "ema_decay": 0.9999,
}

print("Fine-tuning hyperparameters:")
for k, v in FINETUNE_ARGS.items():
    print(f"  {k}: {v}")

In [ ]:
# Run fine-tuning
# Make sure pretraining has completed first

!python run_image_classification.py \
    --model_name_or_path outputs/nepa-tiny-cifar10-pretrain \
    --config_name configs/finetune/nepa-tiny-patch14-224-cifar10-sft \
    --dataset_name cifar10 \
    --load_from_disk False \
    --do_train \
    --do_eval \
    --output_dir outputs/nepa-tiny-cifar10-sft \
    --num_train_epochs 50 \
    --per_device_train_batch_size 64 \
    --per_device_eval_batch_size 128 \
    --learning_rate 5e-4 \
    --lr_scheduler_type cosine \
    --warmup_ratio 0.10 \
    --weight_decay 0.05 \
    --llrd 0.65 \
    --ema_decay 0.9999 \
    --eval_strategy epoch \
    --save_strategy epoch \
    --load_best_model_at_end True \
    --metric_for_best_model eval_accuracy \
    --logging_steps 50 \
    --seed 1337 \
    --bf16 True \
    --dataloader_num_workers 2 \
    --remove_unused_columns False \
    --report_to none

## 6. Evaluation

Evaluate the fine-tuned model on the CIFAR-10 test set.

In [ ]:
# Run evaluation
!python run_image_classification.py \
    --model_name_or_path outputs/nepa-tiny-cifar10-sft \
    --dataset_name cifar10 \
    --load_from_disk False \
    --do_eval \
    --per_device_eval_batch_size 128 \
    --bf16 True \
    --dataloader_num_workers 2 \
    --remove_unused_columns False

In [ ]:
# Interactive inference
from transformers import AutoImageProcessor
import torch

# Load the fine-tuned model
model_path = "outputs/nepa-tiny-cifar10-sft"
model = ViTNepaForImageClassification.from_pretrained(model_path)
processor = AutoImageProcessor.from_pretrained(model_path)

model.eval()
if torch.cuda.is_available():
    model = model.cuda()

print(f"Model loaded from {model_path}")

In [ ]:
# Test on some samples
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i, ax in enumerate(axes.flat):
    sample = dataset['test'][i]
    image = sample['img']
    true_label = CIFAR10_LABELS[sample['label']]
    
    # Inference
    inputs = processor(images=image, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        pred_idx = outputs.logits.argmax(-1).item()
        pred_label = CIFAR10_LABELS[pred_idx]
    
    color = 'green' if pred_label == true_label else 'red'
    ax.imshow(image)
    ax.set_title(f"True: {true_label}\nPred: {pred_label}", color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 7. Save/Load Checkpoints

Instructions for saving checkpoints to Google Drive and loading them later.

In [ ]:
# Mount Google Drive (for saving/loading checkpoints)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Save checkpoint to Google Drive
import shutil

drive_output_dir = "/content/drive/MyDrive/nepa-checkpoints"
os.makedirs(drive_output_dir, exist_ok=True)

# Copy pretrained model
shutil.copytree(
    "outputs/nepa-tiny-cifar10-pretrain",
    f"{drive_output_dir}/nepa-tiny-cifar10-pretrain",
    dirs_exist_ok=True
)

# Copy fine-tuned model
shutil.copytree(
    "outputs/nepa-tiny-cifar10-sft",
    f"{drive_output_dir}/nepa-tiny-cifar10-sft",
    dirs_exist_ok=True
)

print(f"Checkpoints saved to {drive_output_dir}")

In [ ]:
# Load checkpoint from Google Drive (in a new session)
# Uncomment and run after mounting drive

# drive_output_dir = "/content/drive/MyDrive/nepa-checkpoints"
# model = ViTNepaForImageClassification.from_pretrained(
#     f"{drive_output_dir}/nepa-tiny-cifar10-sft"
# )
# print("Model loaded from Google Drive!")

## Notes

- **Pretraining** uses next-embedding prediction (autoregressive learning) without labels
- **Fine-tuning** uses the pretrained weights and trains a classification head
- The tiny model (~15M params) is designed for fast iteration on single GPU
- CIFAR-10 images (32x32) are resized to 224x224 to match the model's expected input size
- For production training, consider using the shell scripts in `scripts/` directory